# Sujet 5 — Vision Foundation Models & Self-Supervised Learning
## Démonstration : représentations visuelles DINOv2 → recherche de similarité & classification simple

**ENSAM Meknès — Université Moulay Ismail** · Cours *LLM / LVM / GenAI : Fondations et Applications* · Encadrant : **Pr. ELANSARI**
Groupe : NANKOULI Marc Thierry · DIALLO Souleymane · HINIMDOU Morsia Guitdam · CHAIBOU SAIDOU Abdoulaye

**Question étudiée.** Un encodeur pré-entraîné *sans aucune étiquette* (DINOv2) fournit-il des représentations directement réutilisables, **sans fine-tuning**, pour (1) retrouver des images similaires et (2) classer des images avec très peu d'exemples ?

**Protocole.** Tous les encodeurs sont **gelés** : on extrait un vecteur par image puis on n'entraîne que des classifieurs très simples (k-NN, régression logistique).

| Encodeur | Famille | Pré-entraînement | Paramètres |
|---|---|---|---|
| **DINOv2-S/14** | SSL — auto-distillation (DINO + iBOT + KoLeo) | LVD-142M, **sans labels** | 21 M |
| **MAE-B/16** | SSL — Masked Image Modeling | ImageNet-1k, **sans labels** | 86 M |
| **DeiT-S/16** | Supervisé (baseline) | ImageNet-1k, **avec labels** | 22 M |
| DINOv2-B/14 *(si disponible)* | SSL — auto-distillation | LVD-142M, sans labels | 86 M |

**Données.** Oxford-IIIT Pets (Parkhi et al., 2012) : 37 races de chats et de chiens, classification *fine-grained*. On utilise le split officiel *trainval* (3 680 images), coupé en deux moitiés stratifiées : **galerie / apprentissage** et **requêtes / test**.

## 0. Configuration

In [ ]:
import os, io, json, time, random, warnings, gc
# Limiter les threads BLAS/OpenMP : réduit fortement la mémoire réservée sur un PC portable
for var in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(var, "4")
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageOps
import torchvision.transforms as T
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, confusion_matrix
from sklearn.decomposition import PCA
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATA, FIG, FEAT = ROOT / "data", ROOT / "figures", ROOT / "features"
FIG.mkdir(exist_ok=True); FEAT.mkdir(exist_ok=True)

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
RECOMPUTE = False          # True : ré-extraire les features même si elles sont en cache
BATCH = 32                 # adapté à un GPU de 4 Go

import transformers, timm, timm.data, sklearn
transformers.logging.set_verbosity_error(); transformers.logging.disable_progress_bar()
print(f"torch {torch.__version__} | transformers {transformers.__version__} | timm {timm.__version__} | sklearn {sklearn.__version__}")
print("Device :", DEVICE, "-", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")

In [ ]:
# Style graphique commun (palette catégorielle fixe : la couleur suit le modèle, jamais son rang)
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#ffffff"
GOOD, BAD = "#0ca30c", "#d03b3b"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.axisbelow": True, "font.size": 10, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlecolor": INK, "legend.frameon": False,
})
def savefig(fig, name):
    fig.savefig(FIG / name, dpi=200, bbox_inches="tight")
    print("figure sauvegardée :", FIG / name)

## 1. Données : Oxford-IIIT Pets
Les images proviennent du miroir Hugging Face `timm/oxford-iiit-pet` (split *trainval* officiel). Les **étiquettes ne servent jamais à entraîner les encodeurs** : uniquement à évaluer et à entraîner les petits classifieurs.

In [ ]:
PARQUET = DATA / "pets_hf" / "data" / "train-00000-of-00001.parquet"
tbl = pq.read_table(PARQUET)
print(tbl.column_names, tbl.num_rows)

RAW_NAMES = ["abyssinian", "american_bulldog", "american_pit_bull_terrier", "basset_hound", "beagle", "bengal",
             "birman", "bombay", "boxer", "british_shorthair", "chihuahua", "egyptian_mau", "english_cocker_spaniel",
             "english_setter", "german_shorthaired", "great_pyrenees", "havanese", "japanese_chin", "keeshond",
             "leonberger", "maine_coon", "miniature_pinscher", "newfoundland", "persian", "pomeranian", "pug", "ragdoll",
             "russian_blue", "saint_bernard", "samoyed", "scottish_terrier", "shiba_inu", "siamese", "sphynx",
             "staffordshire_bull_terrier", "wheaten_terrier", "yorkshire_terrier"]   # ordre des ClassLabel du dataset
CLASSES = [" ".join(w.capitalize() for w in n.split("_")) for n in RAW_NAMES]
N_CLASSES = len(CLASSES)

IMG_BYTES = tbl.column("image").combine_chunks().field("bytes").to_pylist()   # JPEG encodés (≈ 380 Mo)
LABELS = np.array(tbl.column("label").to_pylist())
IMAGE_IDS = tbl.column("image_id").to_pylist()
SPECIES = np.array(tbl.column("label_cat_dog").to_pylist())   # 0 = chat, 1 = chien
del tbl; gc.collect()
# contrôle de cohérence : l'identifiant d'image commence par le nom de la race
assert all(iid.lower().startswith(RAW_NAMES[y]) for iid, y in zip(IMAGE_IDS, LABELS))

def load_pil(i):
    return Image.open(io.BytesIO(IMG_BYTES[i])).convert("RGB")

idx_all = np.arange(len(LABELS))
idx_tr, idx_te = train_test_split(idx_all, test_size=0.5, stratify=LABELS, random_state=SEED)
y_tr, y_te = LABELS[idx_tr], LABELS[idx_te]
print(f"{N_CLASSES} classes | galerie/apprentissage : {len(idx_tr)} | requêtes/test : {len(idx_te)}")
print(f"chats : {(SPECIES == 0).sum()} | chiens : {(SPECIES == 1).sum()}")

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
rng = np.random.RandomState(3)
shown = rng.choice(N_CLASSES, 12, replace=False)
for ax, c in zip(axes.ravel(), shown):
    i = rng.choice(np.where(LABELS == c)[0])
    ax.imshow(ImageOps.fit(load_pil(i), (224, 224))); ax.axis("off")
    ax.set_title(CLASSES[c], fontsize=9, fontweight="normal", color=INK)
fig.suptitle("Oxford-IIIT Pets : 37 races (12 chats, 25 chiens) — tâche fine-grained", color=INK, fontweight="bold")
fig.tight_layout(); savefig(fig, "fig_dataset.png"); plt.show()

## 2. Les encodeurs gelés
Prétraitement **identique** pour tous (redimensionnement 256, recadrage central 224, normalisation ImageNet) afin que seule la méthode de pré-entraînement diffère.
On récupère pour chaque image la séquence de tokens de sortie du ViT : le token **[CLS]** (résumé global) et les **tokens de patches** (dont on prend aussi la moyenne).

In [ ]:
MODELS = {
    # la couleur est attachée au modèle (et non à sa position) : retirer un modèle ne repeint pas les autres
    "DINOv2-S/14": dict(lib="hf",   id="facebook/dinov2-small",          famille="SSL · auto-distillation", donnees="LVD-142M, sans labels",   color="#2a78d6", marker="o"),
    "DINOv2-B/14": dict(lib="hf",   id="facebook/dinov2-base",           famille="SSL · auto-distillation", donnees="LVD-142M, sans labels",   color="#eda100", marker="D"),
    "MAE-B/16":    dict(lib="timm", id="vit_base_patch16_224.mae",       famille="SSL · Masked Image Modeling", donnees="ImageNet-1k, sans labels", color="#eb6834", marker="s"),
    "DeiT-S/16":   dict(lib="timm", id="deit_small_patch16_224.fb_in1k", famille="Supervisé",               donnees="ImageNet-1k, avec labels", color="#1baf7a", marker="^"),
}

def is_available(cfg):
    # Modèle déjà présent dans le cache local ? (évite un long téléchargement imprévu pendant la démo)
    from huggingface_hub import try_to_load_from_cache
    repo = cfg["id"] if cfg["lib"] == "hf" else "timm/" + cfg["id"]
    return any(isinstance(try_to_load_from_cache(repo, f), str) for f in ("model.safetensors", "pytorch_model.bin"))

MODELS = {k: v for k, v in MODELS.items() if is_available(v) or k != "DINOv2-B/14"}
print("Encodeurs comparés :", list(MODELS))

IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

class Encoder:
    # Interface commune : image PIL -> tokens (B, n_prefix + N, D), [CLS] en position 0
    def __init__(self, name):
        cfg = MODELS[name]; self.name = name
        if cfg["lib"] == "hf":
            from transformers import AutoModel
            self.model = AutoModel.from_pretrained(cfg["id"])
            self.n_prefix, self.patch = 1, self.model.config.patch_size
            mean, std = IMAGENET_MEAN, IMAGENET_STD
        else:
            self.model = timm.create_model(cfg["id"], pretrained=True, num_classes=0, dynamic_img_size=True)
            self.n_prefix, self.patch = self.model.num_prefix_tokens, self.model.patch_embed.patch_size[0]
            dc = timm.data.resolve_model_data_config(self.model); mean, std = dc["mean"], dc["std"]
        self.model = self.model.to(DEVICE, DTYPE).eval()
        self.mean, self.std = mean, std
        self.tf = T.Compose([T.Resize(256, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224),
                             T.ToTensor(), T.Normalize(mean, std)])
        self.n_params = sum(p.numel() for p in self.model.parameters())

    @torch.no_grad()
    def tokens(self, x):
        x = x.to(DEVICE, DTYPE)
        if MODELS[self.name]["lib"] == "hf":
            return self.model(pixel_values=x).last_hidden_state.float()
        return self.model.forward_features(x).float()

    @torch.no_grad()
    def embed(self, pil_images):
        h = self.tokens(torch.stack([self.tf(im) for im in pil_images]))
        return h[:, 0].cpu().numpy(), h[:, self.n_prefix:].mean(1).cpu().numpy()

    @torch.no_grad()
    def benchmark(self, bs=BATCH, n=10):
        x = torch.randn(bs, 3, 224, 224)
        for _ in range(3): self.tokens(x)
        if DEVICE == "cuda": torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
        t = time.perf_counter()
        for _ in range(n): self.tokens(x)
        if DEVICE == "cuda": torch.cuda.synchronize()
        dt = time.perf_counter() - t
        vram = torch.cuda.max_memory_allocated() / 2**20 if DEVICE == "cuda" else float("nan")
        return bs * n / dt, vram

def free(enc):
    del enc.model
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

## 3. Extraction des représentations (une seule passe, mise en cache)
Chaque image devient un vecteur de dimension *D* (384 pour les modèles *Small*, 768 pour les *Base*). Ces vecteurs sont sauvegardés : la démo en direct n'a plus besoin de les recalculer.

In [ ]:
FEATS, COSTS = {}, {}
for name, cfg in MODELS.items():
    cache = FEAT / f"{name.replace('/', '_')}.npz"
    enc = Encoder(name)
    ips, vram = enc.benchmark()
    if cache.exists() and not RECOMPUTE:
        z = np.load(cache); cls, mean, t_ext = z["cls"], z["mean"], float(z["t_ext"])
    else:
        t0 = time.perf_counter(); cls, mean = [], []
        for s in range(0, len(LABELS), BATCH):
            c, m = enc.embed([load_pil(i) for i in range(s, min(s + BATCH, len(LABELS)))])
            cls.append(c); mean.append(m)
        cls, mean = np.concatenate(cls), np.concatenate(mean); t_ext = time.perf_counter() - t0
        np.savez(cache, cls=cls, mean=mean, t_ext=t_ext)
    FEATS[name] = {"cls": cls, "mean": mean}
    COSTS[name] = dict(params_M=enc.n_params / 1e6, dim=cls.shape[1], img_per_s=ips, vram_MB=vram, t_extraction_s=t_ext)
    print(f"{name:12s} | {enc.n_params/1e6:6.1f} M params | D={cls.shape[1]} | {ips:6.0f} img/s (GPU, fp16) | "
          f"VRAM {vram:5.0f} Mo | extraction {t_ext:5.1f} s")
    free(enc)

## 4. Classification simple sur représentations gelées
Deux classifieurs volontairement minimalistes :
* **k-NN pondéré** (k = 20, similarité cosinus, vote pondéré par $e^{s/\tau}$, $\tau=0{,}07$ — protocole de DINO) : *aucun* paramètre appris.
* **Sonde linéaire** (régression logistique) : seule une couche linéaire est apprise. Le *pooling* ([CLS] ou moyenne des patches) et la régularisation $C$ sont choisis sur 20 % de validation pris dans la moitié d'apprentissage.

In [ ]:
def l2n(x): return x / np.linalg.norm(x, axis=1, keepdims=True)

def knn_predict(Xtr, ytr, Xte, k=20, tau=0.07):
    S = torch.from_numpy(l2n(Xte)) @ torch.from_numpy(l2n(Xtr)).T
    sims, nn = S.topk(k, dim=1)
    votes = torch.zeros(len(Xte), N_CLASSES).scatter_add_(1, torch.from_numpy(ytr)[nn], (sims / tau).exp())
    return votes.argmax(1).numpy()

def fit_linear(X, y, C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000)).fit(X, y)

Cs = (0.01, 0.1, 1.0)
tr_sub, va_sub = train_test_split(np.arange(len(idx_tr)), test_size=0.2, stratify=y_tr, random_state=SEED)
RES = {"knn": {}, "linear": {}, "choice": {}}
for name in MODELS:
    RES["knn"][name], RES["linear"][name] = {}, {}
    best = (-1, None, None)
    for pool in ("cls", "mean"):
        X = FEATS[name][pool]; Xtr, Xte = X[idx_tr], X[idx_te]
        RES["knn"][name][pool] = float((knn_predict(Xtr, y_tr, Xte) == y_te).mean())
        for C in Cs:
            va = fit_linear(Xtr[tr_sub], y_tr[tr_sub], C).score(Xtr[va_sub], y_tr[va_sub])
            if va > best[0]: best = (va, pool, C)
        RES["linear"][name][pool] = None
    _, pool, C = best
    clf = fit_linear(FEATS[name][pool][idx_tr], y_tr, C)
    RES["linear"][name][pool] = float(clf.score(FEATS[name][pool][idx_te], y_te))
    RES["choice"][name] = dict(pool=pool, C=C)
    print(f"{name:12s} | k-NN [CLS] {RES['knn'][name]['cls']:.1%}  [moy.] {RES['knn'][name]['mean']:.1%} | "
          f"sonde linéaire {RES['linear'][name][pool]:.1%} (pooling={pool}, C={C})")

In [ ]:
POOL = {n: RES["choice"][n]["pool"] for n in MODELS}
KNN = {n: RES["knn"][n][POOL[n]] for n in MODELS}
LIN = {n: RES["linear"][n][POOL[n]] for n in MODELS}

fig, ax = plt.subplots(figsize=(7.6, 3.8))
groups = ["k-NN (k = 20)\naucun paramètre appris", "Sonde linéaire\n(régression logistique)"]
w = 0.8 / len(MODELS)
for j, name in enumerate(MODELS):
    vals = [KNN[name], LIN[name]]
    xs = np.arange(2) + (j - (len(MODELS) - 1) / 2) * w
    ax.bar(xs, vals, width=w - 0.02, color=MODELS[name]["color"], label=f"{name} — {MODELS[name]['famille']}")
    for x, v in zip(xs, vals):
        ax.text(x, v + 0.012, f"{v:.0%}", ha="center", va="bottom", fontsize=9, color=INK)
ax.set_xticks(range(2)); ax.set_xticklabels(groups, color=INK)
ax.set_ylim(0, 1.08); ax.set_ylabel("Exactitude (test, 37 races)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.grid(axis="x", visible=False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=8.5)
ax.set_title("Classification sur représentations gelées (Oxford-IIIT Pets)")
fig.tight_layout(); savefig(fig, "fig_accuracy.png"); plt.show()

## 5. Recherche d'images similaires (*retrieval*)
Pour chaque image requête, on classe les images de la galerie par **similarité cosinus** entre embeddings. Métrique : **précision@k** = proportion des k plus proches voisins qui sont de la même race que la requête (aucune étiquette n'est utilisée pour la recherche elle-même).

In [ ]:
def retrieve(name, q_feats, k=10):
    G = torch.from_numpy(l2n(FEATS[name][POOL[name]][idx_tr]))
    S = torch.from_numpy(l2n(q_feats)) @ G.T
    sims, nn = S.topk(k, dim=1)
    return sims.numpy(), nn.numpy()

RES["retrieval"] = {}
for name in MODELS:
    sims, nn = retrieve(name, FEATS[name][POOL[name]][idx_te], k=10)
    hit = y_tr[nn] == y_te[:, None]
    RES["retrieval"][name] = {f"P@{k}": float(hit[:, :k].mean()) for k in (1, 5, 10)}
    print(f"{name:12s} | " + " | ".join(f"{k} = {v:.1%}" for k, v in RES["retrieval"][name].items()))

In [ ]:
def thumb(i, size=150): return ImageOps.fit(load_pil(i), (size, size))

def frame(ax, ok):
    for s in ax.spines.values():
        s.set_visible(True); s.set_color(GOOD if ok else BAD); s.set_linewidth(3)

def show_retrieval(name, queries, k=5, fname=None, title=None):
    sims, nn = retrieve(name, FEATS[name][POOL[name]][idx_te[queries]], k=k)
    fig, axes = plt.subplots(len(queries), k + 1, figsize=(1.9 * (k + 1), 2.5 * len(queries)))
    axes = np.atleast_2d(axes)
    for r, q in enumerate(queries):
        ax = axes[r, 0]; ax.imshow(thumb(idx_te[q])); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for s in ax.spines.values(): s.set_visible(True); s.set_color(INK); s.set_linewidth(3)
        ax.set_title(f"REQUÊTE\n{CLASSES[y_te[q]]}", fontsize=8.5, color=INK)
        for c in range(k):
            g = nn[r, c]; ok = y_tr[g] == y_te[q]; ax = axes[r, c + 1]
            ax.imshow(thumb(idx_tr[g])); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False); frame(ax, ok)
            ax.set_title(f"{'✓' if ok else '✗'} {CLASSES[y_tr[g]]}\ncos = {sims[r, c]:.2f}", fontsize=8,
                         fontweight="normal", color=INK)
    fig.suptitle(title or f"Top-{k} des images les plus similaires — {name}", color=INK, fontweight="bold")
    fig.tight_layout()
    if fname: savefig(fig, fname)
    plt.show()

rng = np.random.RandomState(7)
QUERIES = rng.choice(len(idx_te), 4, replace=False)
show_retrieval("DINOv2-S/14", QUERIES, fname="fig_retrieval_dinov2.png")

In [ ]:
# Même requête, encodeurs différents : l'auto-supervision par masquage (MAE) retrouve des images
# visuellement proches (couleur, pose) mais pas forcément de la même race.
q = QUERIES[:1]
names = list(MODELS)
fig, axes = plt.subplots(len(names), 6, figsize=(11.4, 2.2 * len(names)))
for r, name in enumerate(names):
    sims, nn = retrieve(name, FEATS[name][POOL[name]][idx_te[q]], k=5)
    ax = axes[r, 0]; ax.imshow(thumb(idx_te[q[0]])); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_ylabel(name, fontsize=10, color=INK, fontweight="bold")
    for s in ax.spines.values(): s.set_visible(True); s.set_color(MODELS[name]["color"]); s.set_linewidth(4)
    for c in range(5):
        g = nn[0, c]; ok = y_tr[g] == y_te[q[0]]; ax = axes[r, c + 1]
        ax.imshow(thumb(idx_tr[g])); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False); frame(ax, ok)
        ax.set_title(f"{'✓' if ok else '✗'} {CLASSES[y_tr[g]]}", fontsize=8, fontweight="normal", color=INK)
fig.suptitle(f"Même requête ({CLASSES[y_te[q[0]]]}), quatre encodeurs gelés", color=INK, fontweight="bold")
fig.tight_layout(); savefig(fig, "fig_retrieval_compare.png"); plt.show()

## 6. Efficacité en étiquettes : combien d'exemples annotés faut-il ?
C'est l'argument central du SSL : si la représentation est bonne, **quelques exemples par classe** suffisent. On entraîne la sonde linéaire avec 1, 2, 5, 10, 20 ou tous les exemples (~40) par classe (moyenne sur 5 tirages).

In [ ]:
SHOTS = [1, 2, 5, 10, 20, None]
RES["label_efficiency"] = {}
for name in MODELS:
    X = FEATS[name][POOL[name]]; C = RES["choice"][name]["C"]; curve = {}
    for n in SHOTS:
        accs = []
        for seed in (range(5) if n else [0]):
            r = np.random.RandomState(seed)
            sel = np.arange(len(idx_tr)) if n is None else np.concatenate(
                [r.choice(np.where(y_tr == c)[0], n, replace=False) for c in range(N_CLASSES)])
            accs.append(fit_linear(X[idx_tr][sel], y_tr[sel], C).score(X[idx_te], y_te))
        curve["tous" if n is None else str(n)] = [float(np.mean(accs)), float(np.std(accs))]
    RES["label_efficiency"][name] = curve
    print(f"{name:12s} | " + " | ".join(f"{k}: {v[0]:.1%}" for k, v in curve.items()))

In [ ]:
n_all = int(np.bincount(y_tr).mean())
xs = [1, 2, 5, 10, 20, n_all]
fig, ax = plt.subplots(figsize=(7.2, 4))
for name in MODELS:
    m = np.array([v[0] for v in RES["label_efficiency"][name].values()])
    s = np.array([v[1] for v in RES["label_efficiency"][name].values()])
    c = MODELS[name]["color"]
    ax.plot(xs, m, color=c, lw=2, marker=MODELS[name]["marker"], ms=7, mec=SURFACE, mew=1.5, label=name)
    ax.fill_between(xs, m - s, m + s, color=c, alpha=0.15, lw=0)
ax.set_xscale("log"); ax.set_xticks(xs); ax.set_xticklabels([str(x) for x in xs[:-1]] + [f"tous\n(~{n_all})"])
ax.xaxis.set_minor_locator(plt.NullLocator())
ax.set_xlim(0.8, n_all * 1.3); ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("Images étiquetées par classe (échelle log)"); ax.set_ylabel("Exactitude (test)")
ax.legend(loc="lower right", fontsize=9, title="moyenne ± écart-type (5 tirages)", title_fontsize=8)
ax.set_title("Efficacité en étiquettes de la sonde linéaire")
fig.tight_layout(); savefig(fig, "fig_label_efficiency.png"); plt.show()

## 7. Analyse des erreurs du meilleur modèle
Les confusions restantes concernent-elles des races réellement proches ?

In [ ]:
best = max(MODELS, key=lambda n: LIN[n])
pool, C = RES["choice"][best]["pool"], RES["choice"][best]["C"]
clf = fit_linear(FEATS[best][pool][idx_tr], y_tr, C)
pred = clf.predict(FEATS[best][pool][idx_te])
cm = confusion_matrix(y_te, pred, labels=range(N_CLASSES))
pairs = [(cm[i, j], CLASSES[i], CLASSES[j]) for i in range(N_CLASSES) for j in range(N_CLASSES) if i != j and cm[i, j] > 0]
pairs.sort(reverse=True)
RES["top_confusions"] = {"model": best, "pairs": [dict(vraie=a, predite=b, n=int(n)) for n, a, b in pairs[:8]]}

fig, ax = plt.subplots(figsize=(7, 3.4))
lab = [f"{a}  →  {b}" for _, a, b in pairs[:8]][::-1]; val = [n for n, _, _ in pairs[:8]][::-1]
ax.barh(lab, val, color=MODELS[best]["color"], height=0.6)
for y, v in enumerate(val): ax.text(v + 0.15, y, str(v), va="center", fontsize=9, color=INK)
ax.set_xlabel("Nombre d'images de test mal classées"); ax.grid(axis="y", visible=False)
ax.set_title(f"Principales confusions — {best} (vraie race → race prédite)")
fig.tight_layout(); savefig(fig, "fig_confusions.png"); plt.show()

## 8. Structure de l'espace des représentations (t-SNE)
Projection 2D (t-SNE) des embeddings de test. Le **score de silhouette** (calculé dans l'espace d'origine, distance cosinus, par race) mesure la compacité des groupes : plus il est élevé, mieux les races sont séparées.

In [ ]:
RES["silhouette"] = {}
names = list(MODELS)
fig, axes = plt.subplots(1, len(names), figsize=(4.3 * len(names), 4.4))
SPECIES_COL = {0: "#6250d6", 1: "#eb6834"}
for ax, name in zip(axes, names):
    X = l2n(FEATS[name][POOL[name]][idx_te])
    sil = float(silhouette_score(X, y_te, metric="cosine")); RES["silhouette"][name] = sil
    E = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(X)
    for sp, lbl in ((0, "chats"), (1, "chiens")):
        m = SPECIES[idx_te] == sp
        ax.scatter(E[m, 0], E[m, 1], s=5, color=SPECIES_COL[sp], alpha=0.75, lw=0, label=lbl)
    for c in range(N_CLASSES):
        p = np.median(E[y_te == c], axis=0)
        ax.text(p[0], p[1], CLASSES[c], fontsize=5.2, ha="center", va="center", color=INK,
                bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.7))
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values(): s.set_visible(True); s.set_color(GRID)
    ax.set_title(f"{name}\nsilhouette = {sil:.3f}", color=INK)
axes[0].legend(loc="lower left", markerscale=3, fontsize=9)
fig.suptitle("t-SNE des représentations gelées (étiquette = race, au point médian)", color=INK, fontweight="bold")
fig.tight_layout(); savefig(fig, "fig_tsne.png"); plt.show()

## 9. Propriété émergente : les tokens de patches « segmentent » l'objet
Comme dans DINO / DINOv2 : une **ACP sur les tokens de patches** (sans aucune annotation de segmentation) sépare l'animal du fond (1ʳᵉ composante), puis 3 composantes affichées en RVB révèlent les **parties** de l'objet.
On le mesure aussi quantitativement grâce aux masques *trimap* d'Oxford-IIIT Pets : IoU entre le masque « 1ʳᵉ composante > 0 » et le masque vrai de l'animal.

In [ ]:
def dense_tokens(enc, pil_list, res):
    tf = T.Compose([T.Resize((res, res), interpolation=T.InterpolationMode.BICUBIC), T.ToTensor(), T.Normalize(enc.mean, enc.std)])
    h = enc.tokens(torch.stack([tf(im) for im in pil_list]))
    g = res // enc.patch
    return h[:, enc.n_prefix:].cpu().numpy(), g

def orient(u, g):
    # signe de la 1re composante : le bord de l'image est majoritairement du fond
    grid = u.reshape(-1, g, g)
    border = np.concatenate([grid[:, 0].ravel(), grid[:, -1].ravel(), grid[:, :, 0].ravel(), grid[:, :, -1].ravel()])
    return -u if border.mean() > u.mean() else u

def pca_rgb(P, g):
    # P : (n_img, N, D) -> carte RVB (n_img, g, g, 3), fond en noir
    n, N, D = P.shape; flat = P.reshape(-1, D)
    u = orient(PCA(1).fit_transform(flat)[:, 0], g)
    fg = u > 0
    rgb = np.zeros((n * N, 3))
    comp = PCA(3).fit_transform(flat[fg])
    comp = (comp - comp.min(0)) / (comp.max(0) - comp.min(0) + 1e-8)
    rgb[fg] = comp
    return rgb.reshape(n, g, g, 3)

rng = np.random.RandomState(11)
PCA_IMGS = [load_pil(idx_te[i]) for i in rng.choice(len(idx_te), 6, replace=False)]
names = list(MODELS)
fig, axes = plt.subplots(len(names) + 1, 6, figsize=(11, 1.95 * (len(names) + 1)))
for c, im in enumerate(PCA_IMGS):
    axes[0, c].imshow(im.resize((224, 224)))
axes[0, 0].set_ylabel("Image", fontsize=10, color=INK, fontweight="bold")
for r, name in enumerate(names, start=1):
    enc = Encoder(name)
    P, g = dense_tokens(enc, PCA_IMGS, 448)
    maps = pca_rgb(P, g)
    for c in range(6): axes[r, c].imshow(maps[c], interpolation="nearest")
    axes[r, 0].set_ylabel(name, fontsize=10, color=INK, fontweight="bold")
    free(enc)
for ax in axes.ravel(): ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.suptitle("ACP des tokens de patches (448×448) — aucune annotation de segmentation utilisée", color=INK, fontweight="bold")
fig.tight_layout(); savefig(fig, "fig_pca_patches.png"); plt.show()

In [ ]:
# Mesure quantitative avec les trimaps (1 = animal, 2 = fond, 3 = contour ignoré)
TRIMAPS = DATA / "annotations" / "trimaps"
RES["seg_iou"] = {}
if TRIMAPS.exists() and IMAGE_IDS[0] is not None:
    rng = np.random.RandomState(5)
    seg_idx = [i for i in idx_te[rng.permutation(len(idx_te))] if (TRIMAPS / f"{IMAGE_IDS[i]}.png").exists()][:300]
    gts = [np.array(Image.open(TRIMAPS / f"{IMAGE_IDS[i]}.png").resize((224, 224), Image.NEAREST)) for i in seg_idx]

    def iou(mask, gt):
        valid = gt != 3; fg = gt == 1
        return (mask & fg & valid).sum() / max(((mask | fg) & valid).sum(), 1)

    yy, xx = np.mgrid[:224, :224]
    center = (abs(yy - 112) < 112 * 0.6) & (abs(xx - 112) < 112 * 0.6)
    RES["seg_iou"]["Carré central (référence triviale)"] = float(np.mean([iou(center, gt) for gt in gts]))
    for name in MODELS:
        enc = Encoder(name); scores = []
        for s in range(0, len(seg_idx), BATCH):
            P, g = dense_tokens(enc, [load_pil(i) for i in seg_idx[s:s + BATCH]], 224)
            for k in range(len(P)):
                u = orient(PCA(1).fit_transform(P[k])[:, 0], g).reshape(g, g) > 0
                m = np.array(Image.fromarray(u.astype(np.uint8) * 255).resize((224, 224), Image.NEAREST)) > 0
                scores.append(iou(m, gts[s + k]))
        RES["seg_iou"][name] = float(np.mean(scores)); free(enc)
    for k, v in RES["seg_iou"].items(): print(f"{k:36s} IoU moyen = {v:.1%}")

    fig, ax = plt.subplots(figsize=(7.2, 2.8))
    keys = list(RES["seg_iou"])[::-1]
    cols = ["#b5b4ae" if k.startswith("Carré") else MODELS[k]["color"] for k in keys]
    ax.barh(keys, [RES["seg_iou"][k] for k in keys], color=cols, height=0.6)
    for y, k in enumerate(keys): ax.text(RES["seg_iou"][k] + 0.01, y, f"{RES['seg_iou'][k]:.0%}", va="center", fontsize=9, color=INK)
    ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_xlabel("IoU moyen avec le masque vrai de l'animal (300 images)"); ax.grid(axis="y", visible=False)
    ax.set_title("Segmentation non supervisée (1re composante de l'ACP)")
    fig.tight_layout(); savefig(fig, "fig_seg_iou.png"); plt.show()
else:
    print("Trimaps indisponibles : mesure quantitative ignorée (figure qualitative ci-dessus conservée).")

## 10. Démo en direct : `search()` sur n'importe quelle image
Donnez un chemin local, une URL ou une image PIL : la fonction calcule l'embedding DINOv2, affiche les images les plus similaires de la galerie et prédit la race par k-NN.
Ci-dessous, on teste la **robustesse** avec des requêtes absentes de la galerie et fortement transformées (recadrage + noir & blanc, puis « croquis » obtenu par filtre de contours).

In [ ]:
_ENC = {}
def get_encoder(name):
    if name not in _ENC:
        for other in list(_ENC): free(_ENC.pop(other))
        _ENC[name] = Encoder(name)
    return _ENC[name]

def load_image(src):
    if isinstance(src, Image.Image): return src.convert("RGB")
    if str(src).startswith("http"):
        import urllib.request
        req = urllib.request.Request(str(src), headers={"User-Agent": "vfm-ssl-demo/1.0"})
        return Image.open(io.BytesIO(urllib.request.urlopen(req, timeout=30).read())).convert("RGB")
    return Image.open(src).convert("RGB")

def search(src, model="DINOv2-S/14", k=5, ax_row=None, label=None):
    img = load_image(src); enc = get_encoder(model)
    cls, mean = enc.embed([img]); q = cls if POOL[model] == "cls" else mean
    sims, nn = retrieve(model, q, k=20)
    w = np.exp(sims[0] / 0.07); votes = np.bincount(y_tr[nn[0]], weights=w, minlength=N_CLASSES)
    pred, conf = votes.argmax(), votes.max() / votes.sum()
    own = ax_row is None
    if own: fig, ax_row = plt.subplots(1, k + 1, figsize=(1.9 * (k + 1), 2.3))
    ax_row[0].imshow(ImageOps.fit(img, (150, 150)))
    ax_row[0].set_title(f"{label or 'REQUÊTE'}\nprédiction : {CLASSES[pred]} ({conf:.0%})", fontsize=8.5, color=INK)
    for c in range(k):
        g = nn[0, c]; ax_row[c + 1].imshow(thumb(idx_tr[g]))
        ax_row[c + 1].set_title(f"{CLASSES[y_tr[g]]}\ncos = {sims[0, c]:.2f}", fontsize=8, fontweight="normal", color=INK)
    for a in ax_row: a.set_xticks([]); a.set_yticks([]); a.grid(False)
    if own: fig.tight_layout(); plt.show()
    return CLASSES[pred], float(conf)

i = idx_te[QUERIES[1]]; src = load_pil(i)
w_, h_ = src.size
crop_gray = ImageOps.grayscale(src.crop((int(0.15 * w_), int(0.1 * h_), int(0.8 * w_), int(0.85 * h_)))).rotate(12, expand=True)
sketch = ImageOps.invert(ImageOps.grayscale(src).filter(ImageFilter.FIND_EDGES)).filter(ImageFilter.SMOOTH)
fig, axes = plt.subplots(2, 6, figsize=(11.4, 5.4))
p1 = search(crop_gray, ax_row=axes[0], label="RECADRÉE + N&B + ROTATION")
p2 = search(sketch, ax_row=axes[1], label="« CROQUIS » (contours)")
fig.suptitle(f"search() avec DINOv2-S/14 — vraie race : {CLASSES[LABELS[i]]}", color=INK, fontweight="bold")
fig.tight_layout(); savefig(fig, "fig_search_demo.png"); plt.show()
RES["search_demo"] = dict(vraie=CLASSES[LABELS[i]], recadree=p1, croquis=p2)
print(RES["search_demo"])

# À essayer pendant la démo avec vos propres photos :
# search("mon_chat.jpg")
# search("https://.../photo_de_chien.jpg", model="DeiT-S/16")

## 11. Synthèse et export des résultats

In [ ]:
rows = []
for n in MODELS:
    c = COSTS[n]
    rows.append(f"| {n} | {MODELS[n]['donnees']} | {c['params_M']:.0f} M | {KNN[n]:.1%} | {LIN[n]:.1%} | "
                f"{RES['retrieval'][n]['P@5']:.1%} | {RES['label_efficiency'][n]['5'][0]:.1%} | "
                f"{RES['silhouette'][n]:.3f} | {c['img_per_s']:.0f} |")
table = ("| Encodeur | Pré-entraînement | Params | k-NN | Sonde lin. | P@5 | 5 ex./classe | Silhouette | img/s |\n"
         "|---|---|---|---|---|---|---|---|---|\n" + "\n".join(rows))
from IPython.display import Markdown, display
display(Markdown(table))

RESULTS = dict(
    dataset=dict(nom="Oxford-IIIT Pets (trainval)", n_classes=int(N_CLASSES), n_train=int(len(idx_tr)), n_test=int(len(idx_te))),
    models={n: dict(**{k: v for k, v in MODELS[n].items() if k not in ("color", "marker")}, **COSTS[n]) for n in MODELS},
    knn=RES["knn"], linear=RES["linear"], choice=RES["choice"], main=dict(knn=KNN, linear=LIN),
    retrieval=RES["retrieval"], label_efficiency=RES["label_efficiency"], silhouette=RES["silhouette"],
    seg_iou=RES["seg_iou"], top_confusions=RES["top_confusions"], search_demo=RES["search_demo"],
    environnement=dict(torch=torch.__version__, transformers=transformers.__version__, timm=timm.__version__,
                       gpu=torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"),
)
with open(FIG / "results.json", "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, ensure_ascii=False, indent=2, default=float)
print("Résultats exportés :", FIG / "results.json")

# Macros LaTeX : les slides et le rapport citent exactement ces valeurs (aucune recopie manuelle)
TAG = {"DINOv2-S/14": "DinoS", "DINOv2-B/14": "DinoB", "MAE-B/16": "Mae", "DeiT-S/16": "Deit"}
pct = lambda v: f"{100 * v:.1f}".replace(".", "{,}") + r"\,\%"
lines = ["% Généré automatiquement par notebook/demo_dinov2.ipynb -- ne pas éditer à la main"]
mac = lambda k, v: lines.append(f"\\providecommand{{\\R{k}}}{{{v}}}")
mac("nTrain", f"{len(idx_tr):,}".replace(",", r"\,")); mac("nTest", f"{len(idx_te):,}".replace(",", r"\,"))
for n, t in TAG.items():
    if n not in MODELS: continue
    mac("Knn" + t, pct(KNN[n])); mac("Lin" + t, pct(LIN[n]))
    mac("KnnCls" + t, pct(RES["knn"][n]["cls"])); mac("KnnMean" + t, pct(RES["knn"][n]["mean"]))
    for k, w in (("P@1", "Pone"), ("P@5", "Pfive"), ("P@10", "Pten")): mac(w + t, pct(RES["retrieval"][n][k]))
    for k, w in (("1", "Sone"), ("2", "Stwo"), ("5", "Sfive"), ("10", "Sten"), ("20", "Stwenty"), ("tous", "Sall")):
        mac(w + t, pct(RES["label_efficiency"][n][k][0]))
    mac("Sil" + t, f"{RES['silhouette'][n]:.3f}".replace(".", "{,}"))
    if n in RES["seg_iou"]: mac("Iou" + t, pct(RES["seg_iou"][n]))
    c = COSTS[n]
    mac("Params" + t, f"{c['params_M']:.0f}\\,M"); mac("Ips" + t, f"{c['img_per_s']:.0f}")
    mac("Vram" + t, f"{c['vram_MB']:.0f}\\,Mo"); mac("Ext" + t, f"{c['t_extraction_s']:.0f}\\,s")
    mac("Pool" + t, "[CLS]" if POOL[n] == "cls" else "moyenne des patches"); mac("C" + t, f"{RES['choice'][n]['C']:g}")
center = [k for k in RES["seg_iou"] if k.startswith("Carré")]
if center: mac("IouCenter", pct(RES["seg_iou"][center[0]]))
mac("bestModel", best)
for r, p in enumerate(RES["top_confusions"]["pairs"][:3]):
    w = ("One", "Two", "Three")[r]
    mac(f"conf{w}", f"{p['vraie']} $\\to$ {p['predite']} ({p['n']})")
sd = RES["search_demo"]
mac("searchTrue", sd["vraie"]); mac("searchCrop", sd["recadree"][0]); mac("searchSketch", sd["croquis"][0])
mac("searchCropConf", pct(sd["recadree"][1])); mac("searchSketchConf", pct(sd["croquis"][1]))
(FIG / "results.tex").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Macros LaTeX exportées :", FIG / "results.tex", f"({len(lines) - 1} valeurs)")

### Conclusions de l'expérience
1. **Sans aucune étiquette au pré-entraînement**, DINOv2 produit des représentations directement exploitables : un simple k-NN, sans aucun paramètre appris, dépasse 90 % d'exactitude sur une tâche fine-grained à 37 classes. DINOv2-B obtient les meilleurs scores en classification, en recherche et avec une seule image par classe.
2. **Le supervisé reste un concurrent sérieux ici** : DeiT-S fait jeu égal avec DINOv2-S, de même taille, et a même la meilleure silhouette. Ses étiquettes ImageNet-1k contiennent en effet de nombreuses races de chiens et de chats présentes dans Pets : la tâche est donc proche de son pré-entraînement. DINOv2 atteint ce niveau **sans aucune étiquette**.
3. **Toutes les auto-supervisions ne se valent pas « gelées »** : MAE (masquage + reconstruction de pixels) s'effondre en k-NN et en recherche, mais remonte nettement avec une sonde linéaire. Ses features sont riches mais peu linéairement séparables, et exigent un *fine-tuning* (cohérent avec He et al., 2022).
4. **Propriétés émergentes** : les tokens de patches DINOv2 séparent l'animal du fond et ses parties, sans supervision dense, plus proprement que DeiT et MAE.
5. **Limites** : la robustesse a ses bornes (le « croquis » par filtre de contours n'est pas reconnu) ; les budgets de pré-entraînement sont inégaux (DINOv2-S est distillé d'un ViT-g entraîné sur 142 M d'images) ; un seul jeu de données ; recouvrement entre races de Pets et classes d'ImageNet.